In [22]:
# importing the necessary libraries
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report
)

In [23]:
# loading and inpecting the data
train = pd.read_csv("IrrigationKaggle/train.csv")

print(train.head())
print()
print(train["Irrigation_Need"].value_counts())
print()
print("missing values:", train.isna().sum().sum())

   id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  \
0   0     Loamy     4.92          32.58            1.01   
1   1      Clay     7.08          56.61            0.44   
2   2      Clay     5.69          27.71            0.81   
3   3     Sandy     5.65          13.32            1.33   
4   4      Clay     7.96          59.14            0.38   

   Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  \
0                     3.05          15.01     50.61       725.99   
1                     2.00          22.92     67.86       985.66   
2                     2.83          26.97     92.22      2201.70   
3                     0.87          13.32     61.57      1357.33   
4                     0.96          20.22     91.11      1538.20   

   Sunlight_Hours  ...  Crop_Type Crop_Growth_Stage  Season Irrigation_Type  \
0            5.90  ...  Sugarcane            Sowing    Zaid            Drip   
1            6.98  ...      Wheat        Vegetative  Kharif         Rainfed   

In [24]:
# performing basic preprocessing
# drop id
train = train.drop("id", axis=1)

# target and features
y_irrigation = train["Irrigation_Need"]
X_irrigation = train.drop("Irrigation_Need", axis=1)

# one-hot encode
X_irrigation = pd.get_dummies(X_irrigation)

# split train/test using only Kaggle train data
X_train_irrigation, X_test_irrigation, y_train_irrigation, y_test_irrigation = train_test_split(
    X_irrigation,
    y_irrigation,
    test_size=0.20,
    stratify=y_irrigation,
    random_state=42
)

In [25]:
# trying out some basic feature engineering 
# simple feature engineering on training set
X_train_irrigation = X_train_irrigation.copy()
X_test_irrigation = X_test_irrigation.copy()

X_train_irrigation["feature_interaction"] = X_train_irrigation.iloc[:, 0] * X_train_irrigation.iloc[:, 1]
X_test_irrigation["feature_interaction"] = X_test_irrigation.iloc[:, 0] * X_test_irrigation.iloc[:, 1]

X_train_irrigation["log_feature"] = np.log1p(X_train_irrigation.iloc[:, 2])
X_test_irrigation["log_feature"] = np.log1p(X_test_irrigation.iloc[:, 2])

X_train_irrigation["ratio_feature"] = X_train_irrigation.iloc[:, 0] / (X_train_irrigation.iloc[:, 2] + 1)
X_test_irrigation["ratio_feature"] = X_test_irrigation.iloc[:, 0] / (X_test_irrigation.iloc[:, 2] + 1)

In [26]:
# creating a helper function for cross-validation
def compare_models_cv(models, X_train, y_train, cv):
    cv_results = []
    for name, model in models.items():
        scores = cross_validate(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring=["accuracy", "f1_macro", "balanced_accuracy"]
        )
        cv_results.append({
            "model": name,
            "cv_accuracy": scores["test_accuracy"].mean(),
            "cv_f1_macro": scores["test_f1_macro"].mean(),
            "cv_balanced_accuracy": scores["test_balanced_accuracy"].mean()
        })
    return pd.DataFrame(cv_results).sort_values("cv_balanced_accuracy", ascending=False)

In [27]:
# defining cross-validation and the naive bayes model
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_irrigation = {
    "Naive Bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("model", GaussianNB())
    ])
}

In [28]:
# running the cross-validation
compare_models_cv(models_irrigation, X_train_irrigation, y_train_irrigation, cv)

,model,cv_accuracy,cv_f1_macro,cv_balanced_accuracy
0,Naive Bayes,0.813117,0.738386,0.804811


In [29]:
# adding the helper function for held-out test evaluation
def evaluate_model_test(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results = {
        "test_accuracy": accuracy_score(y_test, preds),
        "test_precision_macro": precision_score(y_test, preds, average="macro", zero_division=0),
        "test_recall_macro": recall_score(y_test, preds, average="macro", zero_division=0),
        "test_f1_macro": f1_score(y_test, preds, average="macro", zero_division=0),
        "test_balanced_accuracy": balanced_accuracy_score(y_test, preds)
    }

    return model, pd.DataFrame([results])

In [30]:
# evaluate the naive bayes on the help-out test set
trained_nb, test_results_nb = evaluate_model_test(
    models_irrigation["Naive Bayes"],
    X_train_irrigation,
    X_test_irrigation,
    y_train_irrigation,
    y_test_irrigation
)

test_results_nb

,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_balanced_accuracy
0,0.813214,0.702677,0.80532,0.738452,0.80532


In [31]:
# adding a classification report
default_preds_nb = trained_nb.predict(X_test_irrigation)

print("Default classification report:")
print(classification_report(y_test_irrigation, default_preds_nb, zero_division=0))


Default classification report:
              precision    recall  f1-score   support

        High       0.46      0.80      0.59      4202
         Low       0.90      0.84      0.87     73983
      Medium       0.75      0.77      0.76     47815

    accuracy                           0.81    126000
   macro avg       0.70      0.81      0.74    126000
weighted avg       0.83      0.81      0.82    126000



In [32]:
# add predicted probablities
probs_nb = trained_nb.predict_proba(X_test_irrigation)
print("Class order:") 
print(trained_nb.named_steps["model"].classes_)

Class order:
['High' 'Low' 'Medium']


In [33]:
# adding the threshold tuning helper
def evaluate_class_threshold(y_true, probs, class_labels, target_class, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.1, 0.9, 17)

    class_index = list(class_labels).index(target_class)
    rows = []

    baseline_preds = class_labels[np.argmax(probs, axis=1)]

    for t in thresholds:
        preds = np.array(baseline_preds, dtype=object)
        preds[probs[:, class_index] >= t] = target_class

        y_true_binary = (y_true == target_class).astype(int)
        preds_binary = (preds == target_class).astype(int)

        rows.append({
            "threshold": t,
            "precision": precision_score(y_true_binary, preds_binary, zero_division=0),
            "recall": recall_score(y_true_binary, preds_binary, zero_division=0),
            "f1": f1_score(y_true_binary, preds_binary, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_true_binary, preds_binary)
        })

    return pd.DataFrame(rows)

In [34]:
print(y_train_irrigation.value_counts())

Irrigation_Need
Low       295934
Medium    191259
High       16807
Name: count, dtype: int64


In [35]:
target_class = "High"

In [36]:
threshold_results_nb = evaluate_class_threshold(
    y_test_irrigation,
    probs_nb,
    trained_nb.named_steps["model"].classes_,
    target_class
)

threshold_results_nb.sort_values("f1", ascending=False).head(10)

,threshold,precision,recall,f1,balanced_accuracy
8,0.50,0.464222,0.804379,0.588696,0.886175
9,0.55,0.464222,0.804379,0.588696,0.886175
15,0.85,0.464222,0.804379,0.588696,0.886175
14,0.80,0.464222,0.804379,0.588696,0.886175
13,0.75,0.464222,0.804379,0.588696,0.886175
12,0.70,0.464222,0.804379,0.588696,0.886175
11,0.65,0.464222,0.804379,0.588696,0.886175
10,0.60,0.464222,0.804379,0.588696,0.886175
16,0.90,0.464222,0.804379,0.588696,0.886175
7,0.45,0.444198,0.810804,0.573956,0.887902


In [37]:
best = threshold_results_nb.sort_values("f1", ascending=False).iloc[0]
best_threshold = best["threshold"]

class_labels = trained_nb.named_steps["model"].classes_
class_index = list(class_labels).index(target_class)

# default predictions
default_preds = class_labels[np.argmax(probs_nb, axis=1)]

# thresholded predictions
threshold_preds = np.array(default_preds, dtype=object)
threshold_preds[probs_nb[:, class_index] >= best_threshold] = target_class

# binary evaluation for chosen class
y_true_binary = (y_test_irrigation == target_class).astype(int)
default_binary = (default_preds == target_class).astype(int)
threshold_binary = (threshold_preds == target_class).astype(int)

print("Selected class:", target_class)
print("Chosen threshold:", round(best_threshold, 4))
print()

print("Default metrics for selected class")
print("precision:", round(precision_score(y_true_binary, default_binary, zero_division=0), 4))
print("recall:", round(recall_score(y_true_binary, default_binary, zero_division=0), 4))
print("f1:", round(f1_score(y_true_binary, default_binary, zero_division=0), 4))
print("balanced accuracy:", round(balanced_accuracy_score(y_true_binary, default_binary), 4))
print()

print("Thresholded metrics for selected class")
print("precision:", round(precision_score(y_true_binary, threshold_binary, zero_division=0), 4))
print("recall:", round(recall_score(y_true_binary, threshold_binary, zero_division=0), 4))
print("f1:", round(f1_score(y_true_binary, threshold_binary, zero_division=0), 4))
print("balanced accuracy:", round(balanced_accuracy_score(y_true_binary, threshold_binary), 4))

Selected class: High
Chosen threshold: 0.5

Default metrics for selected class
precision: 0.4642
recall: 0.8044
f1: 0.5887
balanced accuracy: 0.8862

Thresholded metrics for selected class
precision: 0.4642
recall: 0.8044
f1: 0.5887
balanced accuracy: 0.8862


In [38]:
print("Thresholded classification report:")
print(classification_report(y_test_irrigation, threshold_preds, zero_division=0))

Thresholded classification report:
              precision    recall  f1-score   support

        High       0.46      0.80      0.59      4202
         Low       0.90      0.84      0.87     73983
      Medium       0.75      0.77      0.76     47815

    accuracy                           0.81    126000
   macro avg       0.70      0.81      0.74    126000
weighted avg       0.83      0.81      0.82    126000



I focused on the High irrigation class because it was the rarest class. I tested multiple thresholds for the High class and found that the best threshold was 0.50, which was the same as the default Naive Bayes. For the High class, the default and thresholded predictions both gave a precision of 0.4642, a recall of 0.8044, an F1 score of 0.5887, and a balanced accuracy of 0.8862.

The default Naive Bayes decision rule was already producing the same predictions as the best tested threshold. This means there was no tradeoff between precision and recall because the predictions did not change.

Compared with my earlier models, Naive Bayes performed worse. My previous LightGBM model achieved much stronger performance, while Naive Bayes worked better as a simple baseline model.